# Block 4 — cross-encoder + context signals

**Reads:** Block 1 records and Block 2 candidates (through `er_eval` / `b2_prod`).
**Writes**, all under `workspace/`:

| output | used by |
|---|---|
| `block4/v1/artifacts/V2/best` — the cross-encoder checkpoint (`intfloat/multilingual-e5-small` + a new 1-logit head) | inference below |
| `block4/v1/ce_logits/<split>/<ROLE>/<COUNTRY>/part-*.parquet` — one raw logit per Block 2 candidate pair | context signals |
| `block4/ctx/v1/<split>/<ROLE>/<COUNTRY>.parquet` — BASE / L0 / L1 / L2 context features | Block 5 |

Weights are downloaded only after a licence gate (MIT / Apache-2.0, ≤ 8B parameters); gradients come
from **CE-FIT only**; CE-DEV only picks the checkpoint; one checkpoint scores every role and the test
set; labels are never features.

Re-running is safe: training is skipped when the checkpoint exists, inference skips every shard with
its `done-*.json`, the context stage skips committed files. `RETRAIN = True` forces a re-train.
Needs a GPU only when something is still missing. Requires Blocks 1–2.

The first run downloads the backbone from Hugging Face into `workspace/hf_cache` (needs internet);
set `$BER_MODEL_DIR` to an already-downloaded copy to run offline.

## Setup (identical first code cell in every notebook)

`BASE` is the folder that holds `src/` — found by walking up from the working directory, or taken
from `$BER_HOME`. Everything then follows: `DATA = BASE/data` (read-only, supplied by the grader),
`WORKSPACE = BASE/workspace` (everything this pipeline writes). `$BER_WORKSPACE` is set to `WORKSPACE`
**before any module is imported**, because `b2_common` / `b2_prod` compute their paths at import
time. `$HF_HOME` keeps model downloads inside the workspace. Nothing outside `BASE` is read or
written except temporary scratch files.

In [ ]:
# Setup — repository layout, workspace, module path
import os
import sys
import tempfile
from pathlib import Path


def find_base(start=None):
    """The repository folder that holds src/. Override with $BER_HOME."""
    if os.environ.get("BER_HOME"):
        return Path(os.environ["BER_HOME"]).expanduser().resolve()
    here = Path(start or os.getcwd()).resolve()
    for d in (here, *here.parents):
        if (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"no folder containing src/ at or above {here}; set BER_HOME")


BASE = find_base()
SRC, DATA, WORKSPACE = BASE / "src", BASE / "data", BASE / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)
ROOT = str(WORKSPACE)

os.environ["BER_WORKSPACE"] = ROOT                       # b2_common / b2_prod read this at import time
os.environ["HF_HOME"] = str(WORKSPACE / "hf_cache")   # model downloads stay inside the workspace
WORK = tempfile.gettempdir()                          # scratch for large local-first writes
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("BASE     :", BASE)
print("DATA     :", DATA, "" if DATA.is_dir() else "(MISSING — supplied by the grader)")
print("WORKSPACE:", WORKSPACE)

import gc
import glob
import importlib
import json
import time

import numpy as np

def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def jdump(obj, path):
    """JSON with numpy types converted, written to a tmp file in the same folder then renamed."""
    def _d(o):
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.bool_):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    path = str(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    json.dump(obj, open(tmp, "w"), indent=1, default=_d)
    os.replace(tmp, path)


def require(*mods):
    """Fail early with one clear message when an expected package is missing."""
    missing = []
    for m in mods:
        try:
            importlib.import_module(m)
        except ImportError:
            missing.append(m)
    if missing:
        raise ImportError(f"missing packages: {missing} — pip install -r ../requirements.txt")

In [ ]:
# Stage 0 — config
B4_VERSION = "v1"
CTX_VERSION = "v1"                  # block4/ctx/<CTX_VERSION>; Block 5 reads the same value
CE_VARIANT = "V2"                   # intfloat/multilingual-e5-small + a new 1-logit head
CE_TAG = "V2"                       # artifacts/<CE_TAG>
TEXT_FORMAT = "A"                   # "{name_norm} | {addr_norm}" — the bi-encoder's text format
TRAIN_C = ["INDIA", "US"]           # labelled countries
TEST_C = ["INDIA", "US", "FRANCE"]  # test countries (France has no labels)
PROD_ROLES = {"train": ["TREE-FIT", "TREE-DEV", "CALIBRATION", "HOLDOUT"], "test": ["TEST"]}
K_CE = None                         # None -> Block 2's K_FINAL (score every retained candidate)
MAX_LEN_OVERRIDE = None             # None -> measured on CE-DEV (99th percentile, multiple of 8, <= 256)
INFER_BATCH = 512
WORKER, N_WORKERS = 0, 1            # >1: run this notebook once per WORKER on separate GPUs
RETRAIN = False
RUN_CE_DEV_REPORT = True            # full CE-DEV metrics for the documentation (cached in reports/)

In [ ]:
# Setup 2 — imports (after BER_WORKSPACE is set)
import pandas as pd
import pyarrow.parquet as pq
require("torch", "transformers", "huggingface_hub")
import er_eval as ev
import b2_prod as b2p
import b4_ce as b4
import b4_ctx as cx

ev.set_root(ROOT)
MAN = b2p.load_merged_manifest()
K_FINAL = int(MAN["k_final"])
K_CE = int(K_CE or K_FINAL)
B4DIR = b4.ver_dir(B4_VERSION)
V2_LOGITS = os.path.join(B4DIR, "ce_logits")
CKPT = os.path.join(B4DIR, "artifacts", CE_TAG, "best")
META = os.path.join(B4DIR, "artifacts", CE_TAG, "meta.json")
CKPT_OK = os.path.exists(os.path.join(CKPT, "config.json")) and os.path.exists(META)
NEED_TRAIN = RETRAIN or not CKPT_OK

import torch
HAS_GPU = torch.cuda.is_available()
log(f"K_FINAL={K_FINAL}  K_CE={K_CE}  checkpoint {'found' if CKPT_OK else 'MISSING'}  GPU={HAS_GPU}  "
    f"b4ctx {cx.CTX_CODE_VERSION}")

In [ ]:
# Stage 1 — preflight: what Block 2 gives us, and what still has to be done
sh = pd.DataFrame(MAN["shards"])
print(sh.groupby(["split", "role", "country"]).size().rename("shards").to_string())


def shards_of(split, role, country):
    return b2p.select_shards(MAN, split, [role], [country])


def n_done(split, role, country):
    d = os.path.join(V2_LOGITS, split, role, country)
    return len(glob.glob(os.path.join(d, "done-*.json")))


JOB_LIST = [(sp, r, c) for sp, cs in (("train", TRAIN_C), ("test", TEST_C)) for r in PROD_ROLES[sp] for c in cs]
PENDING = [(sp, r, c) for sp, r, c in JOB_LIST if n_done(sp, r, c) < len(shards_of(sp, r, c))]
log(f"training needed: {NEED_TRAIN} | inference (split, role, country) groups with missing shards: {len(PENDING)} of {len(JOB_LIST)}")
if (NEED_TRAIN or PENDING) and not HAS_GPU:
    raise RuntimeError("a GPU is required for the missing training / inference work")
plan = {"train": {"roles": PROD_ROLES["train"], "countries": TRAIN_C},
        "test": {"roles": PROD_ROLES["test"], "countries": TEST_C}}
b4.prepare_folders(B4_VERSION, plan, kinds=("ce_logits",))

## Training (skipped when the checkpoint exists)
Pointwise binary cross-entropy on one logit, **CE-FIT pairs only**: every positive of an S1 within its top 50, the
positives Block 2 missed (injected, only those present in the candidate pool), its 15 hardest non-matches within the
top 30, and 3 random deeper non-matches. One epoch, at most 2M pairs. Every 4,000 steps the model is evaluated on 10%
of the CE-DEV S1s and the best S1-macro-F0.5 checkpoint is kept.

In [ ]:
# Stage 2 — model (licence gate), training pairs, max_len
TEXTS = None
if NEED_TRAIN:
    MODEL_PATH, MODEL_META = b4.fetch_model(CE_VARIANT, B4_VERSION)     # refuses anything but MIT / Apache-2.0, > 8B
    TP_PATH = os.path.join(B4DIR, "train_pairs", f"ce_fit_{CE_TAG}.parquet")
    if os.path.exists(TP_PATH):
        PAIRS = pd.read_parquet(TP_PATH)
        log(f"training pairs loaded: {len(PAIRS):,}")
    else:
        PAIRS = b4.build_train_pairs(b2p, TRAIN_C, k_read=50, k_train=30, n_neg_top=15, n_neg_rand=3,
                                     inject_missed=True, seed=42)
        os.makedirs(os.path.dirname(TP_PATH), exist_ok=True)
        PAIRS.to_parquet(TP_PATH + ".tmp", index=False, compression="zstd")
        os.replace(TP_PATH + ".tmp", TP_PATH)
    TEXTS = b4.TextStore("train", TRAIN_C, TEXT_FORMAT)
    DEV_DF = b4.load_pairs(b2p, "train", ["CE-DEV"], TRAIN_C, K_FINAL)
    from transformers import AutoTokenizer
    _tok = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
    _s = DEV_DF.sample(min(50_000, len(DEV_DF)), random_state=0)
    _a, _b, _ = TEXTS.pair_texts(_s)
    LENQ = b4.measure_token_lengths(_tok, _a, _b)
    MAX_LEN = int(MAX_LEN_OVERRIDE or LENQ["suggested_max_len"])
    log("pair token lengths:", LENQ, "-> max_len", MAX_LEN)
    del _tok, _s, _a, _b

In [ ]:
# Stage 3 — train (CE-FIT gradients; CE-DEV only for checkpoint selection)
if NEED_TRAIN:
    SC_DEV_SUB = ev.S1Scorer(ev.load_eval_universe(["CE-DEV"], countries=TRAIN_C,
                                                   s1_frac=b4.TRAIN_DEFAULTS["eval_s1_frac"]))
    BEST_DIR, HIST = b4.train_ce(CE_VARIANT, MODEL_PATH, MODEL_META, PAIRS, TEXTS, DEV_DF, TEXTS, SC_DEV_SUB,
                                 B4_VERSION, cfg={"max_len": MAX_LEN}, run_tag=CE_TAG, fmt=TEXT_FORMAT)
    print(pd.DataFrame(HIST)[["step", "tag", "train_loss", "s1_macro_f05", "pr_auc"]].to_string())
    del PAIRS
    gc.collect()
CKPT_OK = os.path.exists(os.path.join(CKPT, "config.json"))
assert CKPT_OK, f"no checkpoint at {CKPT}"
CE_META = json.load(open(META))
MAX_LEN = int(CE_META["train_config"]["max_len"])
log(f"checkpoint {CKPT} | max_len {MAX_LEN} | trained on {CE_META['n_train_pairs']:,} pairs | "
    f"CE-DEV(10%) S1-macro F0.5 {CE_META['best_dev_sub_s1_macro_f05']:.4f}")

In [ ]:
# Stage 4 — full CE-DEV report (documentation; cached)
REPORT_P = os.path.join(B4DIR, "reports", f"ce_dev_report_{CE_TAG}.json")
if RUN_CE_DEV_REPORT and not os.path.exists(REPORT_P):
    if TEXTS is None:
        TEXTS = b4.TextStore("train", TRAIN_C, TEXT_FORMAT)
    _dev = b4.load_pairs(b2p, "train", ["CE-DEV"], TRAIN_C, K_FINAL)
    _model, _tok, _dv, _amp = b4.load_model(CKPT, for_training=False)
    _sc = ev.S1Scorer(ev.load_eval_universe(["CE-DEV"], countries=TRAIN_C))
    _rep, _f, _ = b4.eval_ce(_model, _tok, _dev, TEXTS, _sc, MAX_LEN, INFER_BATCH, _dv, _amp)
    jdump(_rep, REPORT_P)
    log("CE-DEV (all S1, all countries):", {k: _rep[k] for k in ("s1_macro_f05", "pr_auc", "model_hit@1", "biencoder_hit@1")})
    del _dev, _model, _tok, _sc
    gc.collect()
elif os.path.exists(REPORT_P):
    _rep = json.load(open(REPORT_P))
    log("CE-DEV report (cached):", {k: _rep[k] for k in ("s1_macro_f05", "pr_auc", "model_hit@1", "biencoder_hit@1")})

## Production inference — one logit per Block 2 candidate (rank ≤ K_CE), every role, TEST
One checkpoint scores everything; resumable per Block 2 shard (a shard counts only once its `done-*.json` exists).

In [ ]:
# Stage 5 — score TREE-FIT / TREE-DEV / CALIBRATION / HOLDOUT (train) and TEST (INDIA, US, FRANCE)
_MODEL = None
for sp, cs in (("train", TRAIN_C), ("test", TEST_C)):
    todo = [c for c in cs for r in PROD_ROLES[sp] if (sp, r, c) in PENDING]
    if not todo:
        log(f"inference {sp}: all shards committed, skip")
        continue
    if _MODEL is None:
        _MODEL = b4.load_model(CKPT, for_training=False)
    b4.run_inference(B4_VERSION, b2p, CKPT, CE_TAG, sp, PROD_ROLES[sp], cs, K_CE, worker=WORKER,
                     n_workers=N_WORKERS, fmt=TEXT_FORMAT, max_len=MAX_LEN, batch_size=INFER_BATCH, model=_MODEL)
del _MODEL
gc.collect()

In [ ]:
# Stage 6 — verify: every Block 2 row with rank <= K_CE has exactly one logit
rows = []
for sp, r, c in JOB_LIST:
    exp = 0
    for s in shards_of(sp, r, c):
        exp += pq.read_table(s["cand_path"], columns=["rank"], filters=[("rank", "<=", K_CE)]).num_rows
    got = 0
    d = os.path.join(V2_LOGITS, sp, r, c)
    for dp in glob.glob(os.path.join(d, "done-*.json")):
        got += int(json.load(open(dp))["n_rows"])
    rows.append({"split": sp, "role": r, "country": c, "block2_rows": exp, "logit_rows": got, "ok": exp == got})
COV = pd.DataFrame(rows)
print(COV.to_string(index=False))
if not COV["ok"].all():
    if N_WORKERS > 1:
        log("some shards belong to other workers - run every worker, then this notebook once more")
    raise AssertionError("logit coverage incomplete - see the table above")
jdump(COV.to_dict("records"), os.path.join(B4DIR, "reports", "coverage_check.json"))
log("coverage OK")

## Context signals (CPU, from the logits only)
`b4_ctx` turns the logits into label-free features at four levels: **BASE** (list context of the CE
score), **L0/L1** (they also need a second cross-encoder, which does not exist here — those features
are dropped automatically), and **L2** (who else claims this record: rank / gap / share /
mutual-best across every S1 of the split). Only ranks, gaps and shares are used, so train and test
densities stay comparable — a train role holds a fraction of the S1s while test has all of them
competing (Block 5 re-checks this).

In [ ]:
# Stage 7 — context features
if N_WORKERS > 1 and WORKER != 0:
    log("context features are built by worker 0 only")
else:
    V1_LOGITS = None                                   # no second cross-encoder in the final pipeline
    cx.run_ctx(ROOT, CTX_VERSION, b2p, "train", TRAIN_C, K_CE, V2_LOGITS, V1_LOGITS)
    cx.run_ctx(ROOT, CTX_VERSION, b2p, "test", TEST_C, K_CE, V2_LOGITS, V1_LOGITS)
    log("has_v1 =", cx.ctx_has_v1(ROOT, CTX_VERSION, "train", "TREE-FIT", TRAIN_C),
        "(False is expected: L0/L1 V1 features are dropped in Block 5)")

In [ ]:
# Stage 8 — sanity: logit distributions, France vs the labelled countries (label-free)
for c in TEST_C:
    _d = cx.load_ctx(ROOT, CTX_VERSION, "test", "TEST", [c], columns=["v2_logit"])["v2_logit"]
    _d = _d.dropna()
    print(f"TEST/{c:<7} logit percentiles p1/p50/p99: " +
          " / ".join(f"{np.percentile(_d.sample(min(500_000, len(_d)), random_state=0), q):.2f}" for q in (1, 50, 99)))
print("\nBlock 4 done. Next: 05_block5.ipynb")